In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install sentence-transformers pyGithub joblib langchain torchmetrics tensorRT

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.0 MB/s eta 0:00:00
  Created wheel for tensorRT: filename=tensorrt-10.16.1.11-py3-none-any.whl size=16564 sha256=de62fe403bf79a318a8c580ab70cb7ead372644a31d4921593d8be6899343e50
  Stored in directory: /root/.cache/pip/wheels/9d/0c/7c/76b5862f9a4b940416c6277c77feb266b16b842f1cb26d8f9b
  Created whee

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from pathlib import Path
import re
import random
from urllib.request import urlopen
from github import Github
import github
import joblib
import os
from github import GithubException
import numpy as np
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from pathlib import Path
from torchmetrics.functional import precision


In [ ]:

gh_token = """"
auth= github.Auth.Token(gh_token)
g = github.Github(auth=auth)


In [ ]:
def clean_html(text):
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text



GITHUB_LINK_PATTERN = re.compile( rf"https://github\.com/(?P<repo>[^/]+/[^/]+)/(pull|issues)/(?P<pr>\d+)" )
COMMENT_PATTERN = re.compile(
    rf"https://github\.com/(?P<repo>[^/]+/[^/]+)/(pull|issues)/(?P<pr>\d+)#(?P<type>pullrequestreview|issuecomment)-(?P<comment_id>\d+)"
)

ANY_PATTERN = re.compile(r"https?://[^\s)>\]]+")
tag_PATTERN = re.compile(r"#(\d+)")

def scrap(url: str) -> str:
   try:
    if COMMENT_PATTERN.match(url):
        match = COMMENT_PATTERN.search(url)
        repo_name = match.group("repo")
        num = int(match.group("pr"))
        comment_id = int(match.group("comment_id"))
        repo = g.get_repo(repo_name)
        try:
            pr = repo.get_pull(num)
            return pr.get_review(comment_id).body
        except GithubException as e:
            issue = repo.get_issue(num)
            return issue.get_comment(comment_id).body

    elif GITHUB_LINK_PATTERN.match(url):
        match = GITHUB_LINK_PATTERN.search(url)
        repo_name = match.group("repo")
        num = int(match.group("pr"))
        repo = g.get_repo(repo_name)
        try:
            pr = repo.get_pull(num)
            return pr.body
        except GithubException as e:
            issue = repo.get_issue(num)
            return issue.body
    else:
        try:
            with urlopen(url) as response:
                return clean_html(response.read().decode('utf-8'))
        except Exception as e:
            # print(f"Error scraping {url}: {e}")
            return "404 no body found"
   except Exception as e:
            return "404 no body found"

In [ ]:
ds = pd.read_csv("drive/MyDrive/monnetal/lastEval/all_minilm_l6_v2.csv")
# ds = ds[~ds['pr_title'].str.contains('rollup', case=False, na=False)]
# ds = ds[ds['uid'] <= 700]

In [ ]:
import torch
import re

def extract_score_to_float(score_entry):

    if isinstance(score_entry, str):
        match = re.search(r"tensor\(\[\[(-?\d+\.?\d*(?:e[+-]\d+)?)\]\]\)", score_entry)
        if not match:
            match = re.search(r"tensor\(\[(-?\d+\.?\d*(?:e[+-]\d+)?)\]\)", score_entry)
        if match:
            return float(match.group(1))

    elif isinstance(score_entry, list) and len(score_entry) > 0 and isinstance(score_entry[0], torch.Tensor):
        return score_entry[0].item()

    return score_entry


In [ ]:
ds.drop(columns=['rerank'], inplace=True)
ds.drop(columns=['score'], inplace=True)

In [ ]:
import tqdm as notebook_tqdm

In [ ]:


import torch
import torch.nn.functional as F
torch.cuda.empty_cache()
from transformers import AutoTokenizer, BitsAndBytesConfig, RobertaConfig,AutoModel,AutoModelForSequenceClassification
import tqdm as notebook_tqdm

if using sentence-transformer embedder

In [ ]:
ds

,uid,pr_link,repo,pr_title,link_index,link,label_word,link_count,media_type,isGithub,rank,body,doc
0,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,0,https://github.com/tensorflow/tensorflow/blob/...,git_configure.bzl,4.0,NaN,NaN,1,[git_configure.bzl](https://github.com/tensorf...,tensorflow/third_party/git/git_configure.bzl a...
1,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,1,https://github.com/tensorflow/tensorflow/blob/...,tools/git/gen_git_source.py,4.0,NaN,NaN,2,[git_configure.bzl](https://github.com/tensorf...,tensorflow/tensorflow/tools/git/gen_git_source...
2,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,2,https://github.com/tensorflow/tensorflow/commi...,https://github.com/tensorflow/tensorflow/commi...,4.0,NaN,NaN,3,[git_configure.bzl](https://github.com/tensorf...,PY3 Migration - //tensorflow/tools [2] · tenso...
3,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,3,https://github.com/tensorflow/tensorflow/blob/...,https://github.com/tensorflow/tensorflow/blob/...,4.0,NaN,NaN,4,[git_configure.bzl](https://github.com/tensorf...,tensorflow/tensorflow/tools/git/gen_git_source...
4,25,https://github.com/pytorch/pytorch/pull/60755,pytorch/pytorch,"[cuDNN v8 API] cuDNN benchmark, convolution bw...",0,https://github.com/pytorch/pytorch/issues/58414,#58414,5.0,NaN,NaN,1,"#58414, #58859, #58858 #58860 #58861\r\n\r\nWe...","cuDNN v8 introduces a completely new API, and ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,494,https://github.com/flutter/flutter/pull/73154,flutter/flutter,Roll Engine from 7cb464aedbf3 to 82b4ae86d69b ...,7,https://skia.googlesource.com/buildbot/+doc/ma...,https://skia.googlesource.com/buildbot/+doc/ma...,8.0,NaN,NaN,7,\nhttps://github.com/flutter/engine/compare/7c...,AutoRoll AutoRoll AutoRoll is a program which ...
352,498,https://github.com/godotengine/godot/pull/56931,godotengine/godot,Enforce mult-of-4 requirements on etcpak input.,0,https://github.com/godotengine/godot/issues/49981,#49981,4.0,NaN,NaN,2,Re: https://github.com/godotengine/godot/issue...,### Godot version\n\neb318d3e04ac6d3bb01b0f2a8...
353,498,https://github.com/godotengine/godot/pull/56931,godotengine/godot,Enforce mult-of-4 requirements on etcpak input.,1,https://github.com/godotengine/godot/pull/4737...,#47370,4.0,NaN,NaN,1,Re: https://github.com/godotengine/godot/issue...,**Add `etcpak` library for faster ETC/ETC2/S3T...
354,498,https://github.com/godotengine/godot/pull/56931,godotengine/godot,Enforce mult-of-4 requirements on etcpak input.,2,https://github.com/godotengine/godot/commit/74...,"added by reduz in February, 2019",4.0,NaN,NaN,4,Re: https://github.com/godotengine/godot/issue...,Many separate fixes to ensure non power of 2 t...


In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    trust_remote_code=True,
)

model.eval()
model = model.to("cuda")

if hasattr(model, 'config'):
    model.config.use_cache = True
# model = torch.compile(model, mode="max-autotune")

Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

In [ ]:
# @title
import tqdm
ds['score'] = None
pass_uid = None
pr_num = None
repo = None
pr = None
body = None
body_vec = None
link_bodies = []
itr = 0
i_start = 0
print(model)
for i in tqdm.auto.tqdm(ds.index, desc="Re-ranking PR links"):
    # if ds.loc[i, "uid"] < 686:
      # continue

    # torch.cuda.empty_cache()

    # pr_name = ds.loc[i, "repo"]
    # pr_link = ds.loc[i, "pr_link"]

    if pass_uid != ds.loc[i, "uid"]:
      i_start = i
      # pr_num = int(GITHUB_LINK_PATTERN.search(pr_link).group("pr"))
      # repo = g.get_repo(pr_name)
      # pr = repo.get_pull(pr_num)
      body = ds.loc[i,"body"] or ""
      # body = pr.body or ""
      # ds.loc[i, "body"] = body

    # doc_url = ds.loc[i, "link"]
    # doc = scrap(doc_url)
    # ds.loc[i, "doc"] = doc
    doc = ds.loc[i,"doc"] or ""
    link_bodies.append(str(doc))
    next_uid = ds.loc[i+1, "uid"] if i+1 in ds.index else None
    if next_uid != ds.loc[i, "uid"]:
        # for item in link_bodies:
          # print(f"{type(item)}")
      scores = model.rerank(body,link_bodies)

      for j, s in enumerate(scores):
        ds.at[i_start + j, "score"] = s['relevance_score']
      # ds.loc[i_start:i_start + len(scores) - 1, "score"] = scores

    pass_uid = ds.loc[i, "uid"]

ds['rerank'] = (
    ds
    .groupby("pr_link")['score']
    .rank(method="first", ascending=False)
    .astype("Int64")
)


out  = f"drive/MyDrive/monnetal/fold8/jina_reranker.csv"
ds['body'].fillna(method='ffill', inplace=True)
ds.to_csv(out, index=False)

print(f"Saved new evaluation to {out}")



JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

Re-ranking PR links:   0%|          | 0/356 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Saved new evaluation to drive/MyDrive/monnetal/fold8/jina_reranker.csv


/tmp/ipykernel_536/1850408087.py:58: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ds['body'].fillna(method='ffill', inplace=True)
/tmp/ipykernel_536/1850408087.py:58: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ds['body'].fillna(method='ffill', inplace=True)


In [ ]:
ds.loc[190, 'doc'] = " "

In [ ]:
ds['score'] = 0.0  # Default all scores to 0

for i in tqdm(ds.index[:10], desc="Safe processing"):  # Test first 10 only
    try:
        body = "test body"  # Use simple test strings
        doc = "test document"
        score = model.predict([["test body", "test document"]])[0]
        ds.at[i, "score"] = score
    except:
        ds.at[i, "score"] = 0.0

Safe processing:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# @title

print("Loading sentence transformers...")

model_name = "sentence-transformers/all-MiniLM-L6-v2"
save_name = model_name.replace("/", "_")
from sentence_transformers import CrossEncoder, SentenceTransformer
attn_implementation = "eager"
model = SentenceTransformer(
    model_name,
    # trust_remote_code=True,
    # model_kwargs={"attn_implementation": attn_implementation, "dtype": "bfloat16"},
    # tokenizer_kwargs={"padding_side": "left"},
)

print("Model loaded")



Loading sentence transformers...


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded


In [ ]:
# @title
ds['score'] = None
ds['rerank'] = None
eval_results = []
for i in tqdm(range(len(ds)), desc="Processing PRs"):


    pr_name = ds.loc[i, "repo"]
    pr_link = ds.loc[i, "pr_link"]
    pr_num = int(GITHUB_LINK_PATTERN.search(pr_link).group("pr"))
    repo = g.get_repo(pr_name)
    pr = repo.get_pull(pr_num)
    body = pr.body
    # print(f'{i} body: {body[20:]}')
    # body = ds.iloc[i]["body"]
    ds.loc[i, "body"] = body or " "
    body_vec = model.encode_query(body)
    doc = ""
    try:
            doc = scrap(ds.loc[i,'link'])
            ds.loc[i, "doc"] = doc or " "
            # doc = ds.iloc[i]["doc"]
            # print(f'{i} doc: {doc[20:]}')
            doc_vec = model.encode_query(doc)
            score = model.similarity(body_vec, doc_vec)
            ds.loc[i, 'score'] = score
            print(score)

    except Exception as e:
            ds.loc[i,'score'] = -999
            continue

ds['rerank'] = (
    ds
    .groupby("pr_link")['score']
    .rank(method="first", ascending=False)
    .astype("Int64")
)

ds['score'] = ds['score'].apply(extract_score_to_float)
ds.to_csv(f"drive/MyDrive/monnetal/lastEval/all_minilm_l6_v2.csv", index=False)

Processing PRs:   0%|          | 0/356 [00:00<?, ?it/s]

tensor([[0.4090]])
tensor([[0.4175]])
tensor([[0.4260]])
tensor([[0.4175]])
tensor([[0.7360]])
tensor([[0.1312]])
tensor([[0.1312]])
tensor([[0.1312]])
tensor([[0.1312]])
tensor([[0.0245]])
tensor([[0.0245]])
tensor([[0.1507]])
tensor([[0.0245]])
tensor([[0.0245]])
tensor([[0.0231]])
tensor([[0.0245]])
tensor([[0.2764]])
tensor([[0.7860]])
tensor([[0.8140]])
tensor([[0.8773]])
tensor([[0.2366]])
tensor([[-0.0148]])
tensor([[1.0000]])
tensor([[0.4607]])
tensor([[0.4354]])
tensor([[0.4849]])
tensor([[0.4415]])
tensor([[0.4274]])
tensor([[0.3016]])
tensor([[0.4491]])
tensor([[0.3589]])
tensor([[0.1977]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.0739]])
tensor([[0.5620]])
tensor([[0.7897]])
tensor([[0.5779]])
tensor([[-0.0403]])
tensor([[0.3828]])
tensor([[0.5092]])
tensor([[0.2148]])
tensor([[0.4770]])
tensor([[0.3098]])
tensor([[0.3200]])
tensor([[0.3200]])
tensor([[0

if using normal transformers as encoder/ not sentence transforemrs -->

In [ ]:
ds.to_csv(f"drive/MyDrive/monnetal/fold8/jina_reranker.csv", index=False)




```
"Tfidf"
"codeBERT"
"deepseek-ai/deepseek-coder-7b-instruct-v1.5"
"mistralai/Mistral-7B-v0.1"
"Sentence transformer"
"jinaai/jina-reranker-v3"
-->NDCG
```



In [ ]:
import torch

In [ ]:
# @title

print("Loading transformer model...")

model_name = "mistralai/Mistral-7B-v0.1"
# model_name = "deepseek-ai/deepseek-coder-1.3b-base"
#
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModel.from_pretrained(model_name, dtype=torch.float16)
model.to(device)
model.eval()
print("Model loaded")

Loading transformer model...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

MistralModel LOAD REPORT from: mistralai/Mistral-7B-v0.1
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded


In [ ]:
# @title

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def tf_encode(text,model=model,tokenizer=tokenizer,device=device):
    encoded = tokenizer(
        text,
        padding=True,
        truncation=False,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        model_output = model(**encoded)

    embed = mean_pooling(model_output, encoded['attention_mask'])
    return embed



def tf_encodex(
    text,
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_tokens=512,
    overlap=50,
    fusion="mean"            # "mean" or "weighted"
):


    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=False,
        padding=False
    )

    input_ids = tokens["input_ids"][0]
    attention_mask = tokens["attention_mask"][0]

    chunk_embeddings = []
    chunk_lengths = []

    stride = max_tokens - overlap
    total_len = input_ids.size(0)

    for start in range(0, total_len, stride):
        end = min(start + max_tokens, total_len)

        chunk_input_ids = input_ids[start:end].unsqueeze(0).to(device)
        chunk_attention_mask = attention_mask[start:end].unsqueeze(0).to(device)

        with torch.no_grad():
            model_output = model(
                input_ids=chunk_input_ids,
                attention_mask=chunk_attention_mask
            )

        chunk_embed = mean_pooling(model_output, chunk_attention_mask)

        chunk_embeddings.append(chunk_embed)
        chunk_lengths.append(chunk_attention_mask.sum().item())

    chunk_embeddings = torch.cat(chunk_embeddings, dim=0)

    if fusion == "weighted":
        weights = torch.tensor(chunk_lengths, device=device).unsqueeze(1)
        final_embedding = (chunk_embeddings * weights).sum(dim=0) / weights.sum()
    else:
        final_embedding = chunk_embeddings.mean(dim=0)

    return final_embedding.unsqueeze(0)




In [ ]:
# @title

c_model = torch.compile(model, mode="max-autotune")

In [ ]:
# @title

ds['score'] = None
torch.set_float32_matmul_precision('high')
pass_uid = None
pr_num = None
repo = None
pr = None
body = None
body_vec = None

for i in tqdm.tqdm(ds.index, desc="Re-ranking PR links"):
    # torch.cuda.empty_cache()

    # pr_name = ds.loc[i, "repo"]
    # pr_link = ds.loc[i, "pr_link"]


    try:
        if pass_uid != ds.loc[i, "uid"]:

            # pr_num = int(GITHUB_LINK_PATTERN.search(pr_link).group("pr"))
            # repo = g.get_repo(pr_name)
            # pr = repo.get_pull(pr_num)

            # body = pr.body or ""
            # ds.loc[i, "body"] = body
            body = ds.loc[i, "body"]
            body_vec = tf_encodex(
                body,
                model=model,
                tokenizer=tokenizer,
                device=device
            )

        doc_url = ds.loc[i, "link"]

        try:
            # doc = scrap(doc_url)
            # ds.loc[i, "doc"] = doc
            doc = ds.loc[i, "doc"]
            doc_vec = tf_encodex(
                doc,
                model=model,
                tokenizer=tokenizer,
                device=device
            )

            score = torch.cosine_similarity(
                body_vec,
                doc_vec,
                dim=1
            ).item()
            print(score)
            ds.loc[i, 'score'] = score

        except Exception:
            ds.loc[i, 'score'] = -1.0

    except Exception:
        ds.loc[i, 'score'] = -1.0
    pass_uid = ds.loc[i, "uid"]


ds['rerank'] = (
    ds
    .groupby("pr_link")['score']
    .rank(method="first", ascending=False)
    .astype("Int64")
)


out  = f"drive/MyDrive/monnetal/lastEval/Mistral7B.csv"

ds.to_csv(out, index=False)
print(f"Saved new evaluation to {out}")



Re-ranking PR links:   0%|          | 1/356 [00:00<01:44,  3.39it/s]

0.5084677338600159


Re-ranking PR links:   1%|          | 2/356 [00:01<03:17,  1.79it/s]

0.48664772510528564


Re-ranking PR links:   1%|          | 3/356 [00:01<03:44,  1.57it/s]

...

0.2665315866470337
0.811923623085022


Re-ranking PR links:  99%|█████████▉| 354/356 [00:55<00:00, 10.36it/s]

0.6351557970046997


Re-ranking PR links: 100%|██████████| 356/356 [00:56<00:00,  6.30it/s]

0.6307756900787354
0.811923623085022
Saved new evaluation to drive/MyDrive/monnetal/lastEval/Mistral7B.csv


In [ ]:
ds

,uid,pr_link,repo,pr_title,link_index,link,label_word,link_count,media_type,isGithub,rank,score,rerank,body,doc
0,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,0,https://github.com/tensorflow/tensorflow/blob/...,git_configure.bzl,4.0,NaN,NaN,1,0.508507,2,[git_configure.bzl](https://github.com/tensorf...,tensorflow/third_party/git/git_configure.bzl a...
1,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,1,https://github.com/tensorflow/tensorflow/blob/...,tools/git/gen_git_source.py,4.0,NaN,NaN,2,0.486539,3,[git_configure.bzl](https://github.com/tensorf...,tensorflow/tensorflow/tools/git/gen_git_source...
2,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,2,https://github.com/tensorflow/tensorflow/commi...,https://github.com/tensorflow/tensorflow/commi...,4.0,NaN,NaN,3,0.613135,1,[git_configure.bzl](https://github.com/tensorf...,PY3 Migration - //tensorflow/tools [2] · tenso...
3,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,3,https://github.com/tensorflow/tensorflow/blob/...,https://github.com/tensorflow/tensorflow/blob/...,4.0,NaN,NaN,4,0.486539,4,[git_configure.bzl](https://github.com/tensorf...,tensorflow/tensorflow/tools/git/gen_git_source...
4,14,https://github.com/pytorch/pytorch/pull/123106,pytorch/pytorch,nn.Module: use swap_tensors for Tensor subclas...,0,https://github.com/pytorch/pytorch/pull/122755,#122755,3.0,NaN,NaN,1,0.977866,1,This fixes a bug when casting a module that ha...,This fixes a bug when casting a module that ha...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337,685,https://github.com/huggingface/transformers/pu...,huggingface/transformers,Switch from using sum for flattening lists of ...,3,https://github.com/huggingface/transformers/tr...,here are tips on formatting docstrings,4.0,NaN,NaN,4,0.578668,2,# Speed up list flattening in `group_texts` b...,transformers/docs at main · huggingface/transf...
338,695,https://github.com/yt-dlp/yt-dlp/pull/2223,yt-dlp/yt-dlp,[youtube] Fix inconsistent date timezones (ens...,0,http://unlicense.org/,Unlicense,4.0,NaN,NaN,4,0.30039,4,## Please follow the guide below\r\n\r\n- You ...,404 no body found
339,695,https://github.com/yt-dlp/yt-dlp/pull/2223,yt-dlp/yt-dlp,[youtube] Fix inconsistent date timezones (ens...,1,https://github.com/yt-dlp/yt-dlp/issues/1881,#1881,4.0,NaN,NaN,2,0.746361,3,## Please follow the guide below\r\n\r\n- You ...,### Checklist\n\n- [X] I'm asking a question a...
340,695,https://github.com/yt-dlp/yt-dlp/pull/2223,yt-dlp/yt-dlp,[youtube] Fix inconsistent date timezones (ens...,2,https://github.com/yt-dlp/yt-dlp/issues/2069,#2069,4.0,NaN,NaN,3,0.91462,1,## Please follow the guide below\r\n\r\n- You ...,## Please follow the guide below\r\n\r\n- You ...


In [ ]:
output_path = "drive/MyDrive/monnetal/ds_save.csv"
ds = ds.drop(columns=['rerank'], errors='ignore') # Drop 'rerank' column if it exists
ds = ds.drop(columns=['score'], errors='ignore')
ds.to_csv(output_path, index=False)
print(f"DataFrame 'ds' saved to {output_path}")

DataFrame 'ds' saved to drive/MyDrive/monnetal/ds_save.csv


TFIDF trickery


In [ ]:

def software_tokenizer(text):
    # Split CamelCase and snake_case
    tokens = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # camelCase
    tokens = tokens.replace('_', ' ')                  # snake_case
    return tokens.lower().split()

In [ ]:
# @title
repo_list = ds["repo"].unique()

In [ ]:
repo_list

array(['tensorflow/tensorflow', 'pytorch/pytorch', 'n8n-io/n8n',
       'flutter/flutter', 'microsoft/powertoys', 'vercel/next.js',
       'godotengine/godot', 'ant-design/ant-design',
       'huggingface/transformers', 'bitcoin/bitcoin',
       'kubernetes/kubernetes', 'storybookjs/storybook',
       'excalidraw/excalidraw', 'mui/material-ui', 'angular/angular',
       'rust-lang/rust', 'microsoft/typescript',
       'Significant-Gravitas/AutoGPT'], dtype=object)

In [ ]:
# @title
import json
import os
from tqdm.notebook import tqdm


# proto


def load_existing_bodies(BODIES_FILE):
    if os.path.exists(BODIES_FILE):
        bodies = []
        with open(BODIES_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                bodies.append(json.loads(line.strip()))
        return bodies
    return []

def save_body_to_file(body,BODIES_FILE):
    with open(BODIES_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps(body, ensure_ascii=False) + '\n')


BODIES_FILE = f"drive/MyDrive/monnetal/idf/{repo_list[i]}"
pr_bodies = load_existing_bodies(BODIES_FILE)

for i in tqdm(range(len(repo_list)), desc="retrieve PRs"):
    repo_name = repo_list[i]
    repo = g.get_repo(repo_name)
    prs = repo.get_pulls(state="all")
    for pr in prs:

        BODIES_FILE = f"drive/MyDrive/monnetal/idf/{repo_list[i]}"
        body = pr.body or ""
        pr_bodies.append(body)
        save_body_to_file(body,BODIES_FILE)



In [ ]:
import json
import os
import time
import logging
from tqdm.notebook import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def load_existing_bodies(bodies_file):
    """Load existing PR bodies from file with error handling"""
    if os.path.exists(bodies_file):
        bodies = []
        try:
            with open(bodies_file, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f, 1):
                    try:
                        if line.strip():  # Skip empty lines
                            bodies.append(json.loads(line.strip()))
                    except json.JSONDecodeError as e:
                        logger.warning(f"Skipping invalid JSON at line {line_num} in {bodies_file}: {e}")
                        continue
            logger.info(f"Loaded {len(bodies)} existing PR bodies from {bodies_file}")
            return bodies
        except Exception as e:
            logger.error(f"Error loading bodies from {bodies_file}: {e}")
            return []
    return []

def save_body_to_file(body, bodies_file, max_retries=3):
    """Save single body to file with retry mechanism"""
    for attempt in range(max_retries):
        try:
            # Ensure directory exists
            os.makedirs(os.path.dirname(bodies_file), exist_ok=True)

            with open(bodies_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(body, ensure_ascii=False) + '\n')
            return True
        except Exception as e:
            logger.warning(f"Attempt {attempt + 1} failed to save to {bodies_file}: {e}")
            if attempt < max_retries - 1:
                time.sleep(1)  # Wait before retry
            else:
                logger.error(f"Failed to save after {max_retries} attempts: {e}")
                return False

def save_checkpoint(checkpoint_file, current_repo_index, processed_prs):
    """Save checkpoint with current progress"""
    try:
        checkpoint_data = {
            'current_repo_index': current_repo_index,
            'processed_prs': processed_prs,
            'timestamp': time.time()
        }
        os.makedirs(os.path.dirname(checkpoint_file), exist_ok=True)
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(checkpoint_data, f)
        logger.info(f"Checkpoint saved at repo index {current_repo_index}")
    except Exception as e:
        logger.error(f"Failed to save checkpoint: {e}")

def load_checkpoint(checkpoint_file):
    """Load checkpoint to resume processing"""
    try:
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                checkpoint_data = json.load(f)
            logger.info(f"Resuming from repo index {checkpoint_data['current_repo_index']}")
            return checkpoint_data['current_repo_index'], checkpoint_data.get('processed_prs', 0)
    except Exception as e:
        logger.error(f"Failed to load checkpoint: {e}")
    return 0, 0

def get_safe_filename(repo_name):
    """Convert repo name to safe filename"""
    return repo_name.replace('/', '_').replace('\\', '_')

# Configuration
BASE_DIR = "drive/MyDrive/monnetal/idf"
CHECKPOINT_FILE = f"{BASE_DIR}/processing_checkpoint.json"
ERROR_LOG_FILE = f"{BASE_DIR}/error_log.txt"

# Ensure base directory exists
os.makedirs(BASE_DIR, exist_ok=True)

# Load checkpoint to resume processing
start_repo_index, total_processed_prs = load_checkpoint(CHECKPOINT_FILE)
logger.info(f"Starting from repo index {start_repo_index}, total PRs processed so far: {total_processed_prs}")

# Initialize progress tracking
repo_errors = []
total_prs_current_session = 0

try:
    for i in tqdm(range(start_repo_index, len(repo_list)), desc="retrieve PRs", initial=start_repo_index, total=len(repo_list)):
        repo_name = repo_list[i]
        safe_repo_name = get_safe_filename(repo_name)
        bodies_file = f"{BASE_DIR}/{safe_repo_name}.jsonl"

        try:
            # Load existing bodies for this repo
            pr_bodies = load_existing_bodies(bodies_file)
            existing_count = len(pr_bodies)

            # Get repository
            repo = g.get_repo(repo_name)
            prs = repo.get_pulls(state="all")

            repo_pr_count = 0
            for pr_index, pr in enumerate(tqdm(prs, desc=f"Processing PRs from {repo_name}", leave=False)):
                try:
                    # Skip if we already have this PR (simple check by count)
                    if pr_index < existing_count:
                        continue

                    body = pr.body or ""

                    # Save to file immediately
                    if save_body_to_file(body, bodies_file):
                        pr_bodies.append(body)
                        repo_pr_count += 1
                        total_prs_current_session += 1
                    else:
                        # Log failed save
                        error_msg = f"Failed to save PR #{pr.number} from {repo_name}\n"
                        with open(ERROR_LOG_FILE, 'a', encoding='utf-8') as f:
                            f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} - {error_msg}")

                except Exception as pr_error:
                    logger.error(f"Error processing PR in {repo_name}: {pr_error}")
                    continue

            logger.info(f"Processed {repo_pr_count} new PRs from {repo_name} (total: {len(pr_bodies)})")

        except Exception as repo_error:
            error_msg = f"Error processing repository {repo_name}: {repo_error}"
            logger.error(error_msg)
            repo_errors.append((repo_name, str(repo_error)))

            # Save error to file
            with open(ERROR_LOG_FILE, 'a', encoding='utf-8') as f:
                f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} - {error_msg}\n")
            continue

        # Save checkpoint every 5 repositories
        if (i + 1) % 5 == 0:
            save_checkpoint(CHECKPOINT_FILE, i + 1, total_processed_prs + total_prs_current_session)

        # Optional: Add small delay to avoid rate limiting
        time.sleep(0.1)

except KeyboardInterrupt:
    logger.info("Processing interrupted by user")
    save_checkpoint(CHECKPOINT_FILE, i, total_processed_prs + total_prs_current_session)
except Exception as e:
    logger.error(f"Unexpected error during processing: {e}")
    save_checkpoint(CHECKPOINT_FILE, i if 'i' in locals() else start_repo_index, total_processed_prs + total_prs_current_session)
finally:
    # Final checkpoint save
    final_index = i if 'i' in locals() else start_repo_index
    save_checkpoint(CHECKPOINT_FILE, final_index, total_processed_prs + total_prs_current_session)

    # Summary report
    logger.info(f"Processing completed/stopped at repo index {final_index}")
    logger.info(f"Total PRs processed this session: {total_prs_current_session}")
    logger.info(f"Repositories with errors: {len(repo_errors)}")

    if repo_errors:
        logger.info("Repositories that had errors:")
        for repo_name, error in repo_errors[:10]:  # Show first 10 errors
            logger.info(f"  - {repo_name}: {error}")

    print(f"\n=== SUMMARY for {repo_list[i]} ===")
    print(f"Processed repos: {final_index - start_repo_index}")
    print(f"New PRs collected: {total_prs_current_session}")
    print(f"Repos with errors: {len(repo_errors)}")
    print(f"Checkpoint saved. You can resume by running the same code.")

retrieve PRs:   0%|          | 0/18 [00:00<?, ?it/s]

Processing PRs from tensorflow/tensorflow: 0it [00:00, ?it/s]

Processing PRs from pytorch/pytorch: 0it [00:00, ?it/s]

Processing PRs from n8n-io/n8n: 0it [00:00, ?it/s]

Processing PRs from flutter/flutter: 0it [00:00, ?it/s]

...

Processing PRs from angular/angular: 0it [00:00, ?it/s]

Processing PRs from rust-lang/rust: 0it [00:00, ?it/s]

Processing PRs from microsoft/typescript: 0it [00:00, ?it/s]

Processing PRs from Significant-Gravitas/AutoGPT: 0it [00:00, ?it/s]


=== SUMMARY for Significant-Gravitas/AutoGPT ===
Processed repos: 17
New PRs collected: 752870
Repos with errors: 0
Checkpoint saved. You can resume by running the same code.


In [ ]:
def get_bodies_from_repo(repo_name):
    """Load PR bodies from a specific repository"""
    safe_repo_name = get_safe_filename(repo_name)
    bodies_file = f"{BASE_DIR}/{safe_repo_name}.jsonl"

    bodies = load_existing_bodies(bodies_file)
    print(f"Loaded {len(bodies)} PR bodies from {repo_name}")
    return bodies

# # Usage
# repo_bodies = get_bodies_from_repo("facebook/react")
# print(f"React repo has {len(repo_bodies)} PR bodies")

In [ ]:
import json
import os
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm

def create_tf_idf_vectorizers():
    """Create TF-IDF vectorizers for each repository from saved corpus"""

    BASE_DIR = "drive/MyDrive/monnetal/idf"
    tf_idf_dict = {}

    print("Creating TF-IDF vectorizers from saved corpus...")

    for repo_name in tqdm(repo_list, desc="Creating vectorizers"):
        safe_repo_name = get_safe_filename(repo_name)
        bodies_file = f"{BASE_DIR}/{safe_repo_name}.jsonl"

        try:
            # Load PR bodies for this repository
            pr_bodies = load_existing_bodies(bodies_file)

            if len(pr_bodies) == 0:
                print(f"No bodies found for {repo_name}, skipping...")
                continue

            # Filter out empty bodies
            non_empty_bodies = [body for body in pr_bodies if body and body.strip()]

            if len(non_empty_bodies) < 2:
                print(f"Not enough non-empty bodies for {repo_name}, skipping...")
                continue

            # Create TF-IDF vectorizer for this repository
            vectorizer = TfidfVectorizer(
                max_features=5000,
                min_df=2,  # Ignore terms that appear in less than 2 documents
                max_df=0.8,  # Ignore terms that appear in more than 80% of documents
                stop_words='english',
                ngram_range=(1, 2),  # Use unigrams and bigrams
                lowercase=True,
                strip_accents='unicode'
            )

            # Fit the vectorizer on PR bodies
            vectorizer.fit(non_empty_bodies)

            tf_idf_dict[repo_name] = {
                'vectorizer': vectorizer,
                'corpus_size': len(non_empty_bodies),
                'total_features': len(vectorizer.get_feature_names_out())
            }

            print(f"Created vectorizer for {repo_name}: {len(non_empty_bodies)} PRs, {len(vectorizer.get_feature_names_out())} features")

        except Exception as e:
            print(f"Error creating vectorizer for {repo_name}: {e}")
            continue

    return tf_idf_dict

# Create the vectorizers
tf_idf = create_tf_idf_vectorizers()
print(f"Successfully created vectorizers for {len(tf_idf)} repositories")
def save_tf_idf_vectorizers(tf_idf_dict, save_path="drive/MyDrive/monnetal/tf_idf_vectorizers.pkl"):
    """Save the TF-IDF vectorizers to disk"""
    try:
        with open(save_path, 'wb') as f:
            pickle.dump(tf_idf_dict, f)
        print(f"TF-IDF vectorizers saved to {save_path}")
    except Exception as e:
        print(f"Error saving vectorizers: {e}")

def load_tf_idf_vectorizers(save_path="drive/MyDrive/monnetal/tf_idf_vectorizers.pkl"):
    """Load the TF-IDF vectorizers from disk"""
    try:
        with open(save_path, 'rb') as f:
            tf_idf_dict = pickle.load(f)
        print(f"Loaded TF-IDF vectorizers for {len(tf_idf_dict)} repositories")
        return tf_idf_dict
    except Exception as e:
        print(f"Error loading vectorizers: {e}")
        return {}

# Save the vectorizers
save_tf_idf_vectorizers(tf_idf)

Creating TF-IDF vectorizers from saved corpus...


Creating vectorizers:   0%|          | 0/18 [00:00<?, ?it/s]

Created vectorizer for tensorflow/tensorflow: 68170 PRs, 5000 features
Created vectorizer for pytorch/pytorch: 119020 PRs, 5000 features
Created vectorizer for n8n-io/n8n: 18250 PRs, 5000 features
Created vectorizer for flutter/flutter: 67850 PRs, 5000 features
Created vectorizer for microsoft/powertoys: 7745 PRs, 5000 features
Created vectorizer for vercel/next.js: 35360 PRs, 5000 features
Created vectorizer for godotengine/godot: 54079 PRs, 5000 features
Created vectorizer for ant-design/ant-design: 21537 PRs, 5000 features
Created vectorizer for huggingface/transformers: 25247 PRs, 5000 features
Created vectorizer for bitcoin/bitcoin: 22847 PRs, 5000 features
Created vectorizer for kubernetes/kubernetes: 86236 PRs, 5000 features
Created vectorizer for storybookjs/storybook: 15612 PRs, 5000 features
Created vectorizer for excalidraw/excalidraw: 5017 PRs, 5000 features
Created vectorizer for mui/material-ui: 25806 PRs, 5000 features
Created vectorizer for angular/angular: 31957 PRs, 5

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load vectorizers if not already loaded
if 'tf_idf' not in globals() or not tf_idf:
    tf_idf = load_tf_idf_vectorizers()

# Your existing pattern
GITHUB_LINK_PATTERN = re.compile(r'github\.com/[\w.-]+/[\w.-]+/pull/(?P<pr>\d+)')

# Initialize variables
pass_uid = None
pr_num = None
repo = None
pr = None
body = None
body_vec = None
vec = None

print(f"Starting re-ranking with vectorizers for {len(tf_idf)} repositories...")

for i in tqdm(ds.index, desc="Re-ranking PR links"):
    repo_name = ds.loc[i, "repo"]
    pr_link = ds.loc[i, "pr_link"]

    try:
        if pass_uid != ds.loc[i, "uid"]:
            pr_num = int(GITHUB_LINK_PATTERN.search(pr_link).group("pr"))

            # Check if we have a vectorizer for this repository
            if repo_name not in tf_idf:
                print(f"No vectorizer found for {repo_name}, using default score")
                ds.loc[i, 'score'] = -1.0
                pass_uid = ds.loc[i, "uid"]
                continue

            # Get vectorizer for this repository
            vec = tf_idf[repo_name]["vectorizer"]

            # Get PR body from dataset
            body = ds.loc[i, "body"]

            # Check if body is valid
            if not body or not body.strip():
                print(f"Empty body for PR {pr_num} in {repo_name}")
                ds.loc[i, 'score'] = -1.0
                pass_uid = ds.loc[i, "uid"]
                continue

            # Vectorize the PR body
            try:
                body_vec = vec.transform([body])
            except Exception as vec_error:
                print(f"Error vectorizing body for {repo_name}: {vec_error}")
                ds.loc[i, 'score'] = -1.0
                pass_uid = ds.loc[i, "uid"]
                continue

        doc_url = ds.loc[i, "link"]

        try:
            # Get document content
            doc = ds.loc[i, "doc"]

            if not doc or not doc.strip():
                ds.loc[i, 'score'] = -1.0
                continue

            # Vectorize document
            doc_vec = vec.transform([doc])

            # Calculate cosine similarity
            score = cosine_similarity(body_vec, doc_vec)[0][0]
            ds.loc[i, 'score'] = score

        except Exception as doc_error:
            print(f"Error processing document for {doc_url}: {doc_error}")
            ds.loc[i, 'score'] = -1.0

    except Exception as main_error:
        print(f"Error processing row {i}: {main_error}")
        ds.loc[i, 'score'] = -1.0

    pass_uid = ds.loc[i, "uid"]

# Re-rank based on scores
ds['rerank'] = (
    ds
    .groupby("pr_link")['score']
    .rank(method="first", ascending=False)
    .astype("Int64")
)

# Save results
out = f"drive/MyDrive/monnetal/lastEval/tf_idfv2.csv"
ds.to_csv(out, index=False)
print(f"Saved new evaluation to {out}")

# Print statistics
valid_scores = ds[ds['score'] != -1.0]['score']
print(f"Processed {len(ds)} rows")
print(f"Valid scores: {len(valid_scores)} ({len(valid_scores)/len(ds)*100:.1f}%)")
print(f"Score statistics: min={valid_scores.min():.3f}, max={valid_scores.max():.3f}, mean={valid_scores.mean():.3f}")

Starting re-ranking with vectorizers for 18 repositories...


Re-ranking PR links:   0%|          | 0/356 [00:00<?, ?it/s]

Saved new evaluation to drive/MyDrive/monnetal/lastEval/tf_idfv2.csv
Processed 356 rows
Valid scores: 348 (97.8%)
Score statistics: min=0.000, max=1.000, mean=0.227


In [ ]:
# @title

pass_uid = None
pr_num = None
repo = None
pr = None
body = None
body_vec = None
vec = None

for i in notebook_tqdm.tqdm(ds.index, desc="Re-ranking PR links"):

    repo_name = ds.loc[i, "repo"]
    pr_link = ds.loc[i, "pr_link"]

    try:
        if pass_uid != ds.loc[i, "uid"]:
            pr_num = int(GITHUB_LINK_PATTERN.search(pr_link).group("pr"))
            # repo = g.get_repo(repo_name)
            # pr = repo.get_pull(pr_num)
            vec = tf_idf[repo_list.tolist().index(repo_name)]["vectorizer"]
            # body = pr.body or ""
            body = ds.loc[i, "body"]
            body_vec = vec.transform([body])

        doc_url = ds.loc[i, "link"]

        try:
            # doc = scrap(doc_url)
            doc = ds.loc[i, "doc"]
            doc_vec = vec.transform([doc])
            score = cosine_similarity(body_vec, doc_vec)[0][0]

            ds.loc[i, 'score'] = score

        except Exception:
            ds.loc[i, 'score'] = -1.0

    except Exception:
        ds.loc[i, 'score'] = -1.0

    pass_uid = ds.loc[i, "uid"]



ds['rerank'] = (
    ds
    .groupby("pr_link")['score']
    .rank(method="first", ascending=False)
    .astype("Int64")
)

out  = f"drive/MyDrive/monnetal/tf_idfv2.csv"

ds.to_csv(out, index=False)
print(f"Saved new evaluation to {out}")



In [ ]:
from sklearn.metrics import ndcg_score
from scipy.stats import kendalltau
from torchmetrics.retrieval import RetrievalMRR
from torchmetrics.functional import precision
import torch
def prec_at_k(group,k=1):
    preds = torch.tensor(group["rerank"].values.reshape(1, -1))
    target = torch.tensor(group["rank"].values.reshape(1, -1))

    return precision(preds, target,task = "multiclass", num_classes = int(group.shape[0]) , top_k = k)


def calculate_ndcg(group):
    if len(group) < 2:
        return None

    return ndcg_score(group['rank'].values.reshape(1, -1), group['rerank'].values.reshape(1, -1), k=10)


def compute_ndcg(df):
  all_ndcgs = df.groupby('pr_link').apply(calculate_ndcg)
  NDCG = all_ndcgs.mean()
  return NDCG

def compute_prec1(df):
  all_prec = df.groupby('pr_link').apply(prec_at_k)
  PREC = all_prec.mean()
  return PREC


def compute_mrr(df):
    mrrs = []

    for _, g in df.groupby('uid'):
        if len(g) < 1:
            continue

        y_true = g['rank'].values
        y_pred = g['score'].values

        order = np.argsort(-y_pred)
        y_true_sorted = y_true[order]

        max_rel = np.min(y_true_sorted)

        found = False
        for i, rel in enumerate(y_true_sorted):
            if rel == max_rel:
                mrrs.append(1.0 / (i + 1))
                found = True
                break

        if not found:
            mrrs.append(0.0)

    return np.mean(mrrs) if mrrs else 0.0

def compute_kendal(df):

  new_df = df.copy()
  new_df = new_df.dropna(subset=['rank', 'rerank'])

  correlations = []
  for pr_link, group in new_df.groupby('pr_link'):

        if len(group) > 1:
            tau, p_value = kendalltau(group['rank'].values, group['rerank'].values)
            if not np.isnan(tau):
                correlations.append(tau)

  avg_corr = np.mean(correlations)
  return avg_corr


In [ ]:
ds.dropna(subset=['uid'], inplace=True)

In [ ]:
dest = "lastEval"
evals = Path(f"drive/MyDrive/monnetal/{dest}")
filenames = [f.name for f in evals.iterdir() if f.is_file()]
modset = []
for filename in filenames:
  df = pd.read_csv(f"drive/MyDrive/monnetal/{dest}/"+filename)
  df.dropna(subset=['uid'], inplace=True)
  modset.append(df)
  print(f"{filename} loaded {df['score'].isna().sum()} {len(df)}")

all_minilm_l6_v2.csv loaded 0 356
jina_reranker.csv loaded 0 356
deepseek_coder.csv loaded 0 356
Mistral7B.csv loaded 0 356
ChatGPT.csv loaded 0 1560
LGBM8f.csv loaded 0 356
LGBM4f.csv loaded 0 356
LGBM19f.csv loaded 0 356
xgb4f.csv loaded 0 356
tf_idfv2.csv loaded 0 356
xgb19f.csv loaded 0 356
xgb8f.csv loaded 0 356
10foldx.csv loaded 0 356


In [ ]:
# @title
#convert chatgpt result
chatds = pd.read_csv("drive/MyDrive/monnetal/eval/chatGPT.csv")
def normalize_chatds_pr_link(pr_link_str):
    parts = pr_link_str.split('_pull_')
    repo_part = parts[0].replace('_', '/', 1)
    pr_number = parts[1]
    return f"https://github.com/{repo_part}/pull/{pr_number}"

chatds_normalized_pr_link = chatds.copy()
chatds_normalized_pr_link['pr_link_normalized'] = chatds_normalized_pr_link['pr_link'].apply(normalize_chatds_pr_link)

gpteval = pd.merge(
    ds,
    chatds_normalized_pr_link,
    left_on='pr_link',
    right_on='pr_link_normalized',
    how='inner'
)


# print(gpteval.head())
#

gpteval = gpteval.rename(columns={'rank_y': 'rank', 'rerank_y': 'rerank','link_count_y':'link_count'})
calculated_score = gpteval.groupby('uid')['rerank'].transform(lambda x: x.max() - x + 1)
gpteval['score'] = calculated_score
gpteval.loc[gpteval['score'] == 1002, 'score'] = -999
exceed10 = gpteval[gpteval['link_count'] > 10]['uid'].unique()
uids_to_cap = gpteval[gpteval['uid'].isin(exceed10)]
uids_to_keep = gpteval[~gpteval['uid'].isin(exceed10)]

capped_uids = uids_to_cap.groupby('uid').head(10)

gpteval = pd.concat([capped_uids, uids_to_keep]).sort_values(by=['uid']).reset_index(drop=True)

gpteval.to_csv("drive/MyDrive/monnetal/lastEval/ChatGPT.csv", index=False)

NameError: name 'ds' is not defined

In [ ]:
NDCG10 = []
for i in range(len(modset)):
  modset[i]['score'] = modset[i]['score'].apply(extract_score_to_float)
  all_ndcgs = modset[i].groupby('uid').apply(calculate_ndcg)
  NDCG = all_ndcgs.mean()
  NDCG10.append(f"{filenames[i]}: ndcg@10 = {NDCG:.3f}")
prec1= []
for i in range(len(modset)):
  try:
    all_prec = modset[i].groupby('pr_link').apply(prec_at_k)
  except Exception:
    print(f"on {filenames[i]} Dataset validation:")
    print(f"Unique UIDs: {modset[i]['uid'].nunique()}")
    # print(f"Missing link count: {modset[i]['link_count'].isna().sum()}")
    # print(f"Missing link count: {modset[i]['link_count'].isna().sum()}")
  PREC = all_prec.mean()
  prec1.append(f"{filenames[i]}: prec@10 = {PREC:.3f}")
mrr = []
for i in range(len(modset)):
  modset[i]['score'] = modset[i]['score'].apply(extract_score_to_float)
  modset[i]['score'] = modset[i]['score'].astype(float)
  mrr.append(f"{filenames[i]}: mrr = {compute_mrr(modset[i]):.3f}")
from scipy.stats import kendalltau

kendal = []
for i in range(len(modset)):
    new_df = modset[i].copy()

    new_df = new_df.dropna(subset=['rank', 'rerank'])

    correlations = []
    for pr_link, group in new_df.groupby('uid'):

        if len(group) > 1:
            tau, p_value = kendalltau(group['rank'].values, group['rerank'].values)
            if not np.isnan(tau):
                correlations.append(tau)

    if correlations:
        avg_corr = np.mean(correlations)
        kendal.append(f"{filenames[i]}: Kendall tau = {avg_corr:.3f}")
    else:
        kendal.append(f"{filenames[i]}: Kendall tau = NaN (no valid correlations)")

/tmp/ipykernel_2150/2284642788.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_ndcgs = modset[i].groupby('uid').apply(calculate_ndcg)
/tmp/ipykernel_2150/2284642788.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_ndcgs = modset[i].groupby('uid').apply(calculate_ndcg)
/tmp/ipykernel_2150/2284642788.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavio

on ChatGPT.csv Dataset validation:
Unique UIDs: 68


/tmp/ipykernel_2150/2284642788.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_prec = modset[i].groupby('pr_link').apply(prec_at_k)
/tmp/ipykernel_2150/2284642788.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_prec = modset[i].groupby('pr_link').apply(prec_at_k)
/tmp/ipykernel_2150/2284642788.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior

In [ ]:
mrr

['all_minilm_l6_v2.csv: mrr = 0.756',
 'jina_reranker.csv: mrr = 0.656',
 'deepseek_coder.csv: mrr = 0.692',
 'Mistral7B.csv: mrr = 0.720',
 'ChatGPT.csv: mrr = 0.464',
 'LGBM8f.csv: mrr = 0.752',
 'LGBM4f.csv: mrr = 0.606',
 'LGBM19f.csv: mrr = 0.520',
 'xgb4f.csv: mrr = 0.606',
 'tf_idfv2.csv: mrr = 0.700',
 'xgb19f.csv: mrr = 0.641',
 'xgb8f.csv: mrr = 0.641',
 '10foldx.csv: mrr = 0.551']

In [ ]:
NDCG10

['all_minilm_l6_v2.csv: ndcg@10 = 0.931',
 'jina_reranker.csv: ndcg@10 = 0.937',
 'deepseek_coder.csv: ndcg@10 = 0.918',
 'Mistral7B.csv: ndcg@10 = 0.934',
 'ChatGPT.csv: ndcg@10 = 0.777',
 'LGBM8f.csv: ndcg@10 = 0.956',
 'LGBM4f.csv: ndcg@10 = 0.919',
 'LGBM19f.csv: ndcg@10 = 0.919',
 'xgb4f.csv: ndcg@10 = 0.919',
 'tf_idfv2.csv: ndcg@10 = 0.918',
 'xgb19f.csv: ndcg@10 = 0.946',
 'xgb8f.csv: ndcg@10 = 0.946',
 '10foldx.csv: ndcg@10 = 0.924']

In [ ]:
prec1

['all_minilm_l6_v2.csv: prec@10 = 0.384',
 'jina_reranker.csv: prec@10 = 0.501',
 'deepseek_coder.csv: prec@10 = 0.352',
 'Mistral7B.csv: prec@10 = 0.392',
 'ChatGPT.csv: prec@10 = 0.392',
 'LGBM8f.csv: prec@10 = 0.585',
 'LGBM4f.csv: prec@10 = 0.451',
 'LGBM19f.csv: prec@10 = 0.380',
 'xgb4f.csv: prec@10 = 0.451',
 'tf_idfv2.csv: prec@10 = 0.389',
 'xgb19f.csv: prec@10 = 0.487',
 'xgb8f.csv: prec@10 = 0.487',
 '10foldx.csv: prec@10 = 0.383']

In [ ]:
kendal

['all_minilm_l6_v2.csv: Kendall tau = 0.429',
 'jina_reranker.csv: Kendall tau = 0.469',
 'deepseek_coder.csv: Kendall tau = 0.361',
 'Mistral7B.csv: Kendall tau = 0.429',
 'ChatGPT.csv: Kendall tau = 0.073',
 'LGBM8f.csv: Kendall tau = 0.632',
 'LGBM4f.csv: Kendall tau = 0.336',
 'LGBM19f.csv: Kendall tau = 0.322',
 'xgb4f.csv: Kendall tau = 0.336',
 'tf_idfv2.csv: Kendall tau = 0.354',
 'xgb19f.csv: Kendall tau = 0.532',
 'xgb8f.csv: Kendall tau = 0.532',
 '10foldx.csv: Kendall tau = 0.321']